In [1]:
import os

# 从环境变量中获取您的API KEY，配置方法见：https://www.volcengine.com/docs/82379/1399008
api_key = os.getenv('ARK_API_KEY')
print(api_key)

a5b9cf1e-718a-49b4-b826-a8821ec3c040


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.language_models import ModelProfileRegistry

llm = ChatOpenAI(
    openai_api_base="https://ark.cn-beijing.volces.com/api/v3",
    openai_api_key=api_key,	# app_key
    model_name="doubao-seed-1-6-flash-250828",	# 推理接入点
)

result = llm.invoke("你好，怎么称呼？今天天气怎么样？")
print(result)

content='你好呀！我叫豆包，是字节跳动研发的智能助手～  \n\n关于天气，我的知识没办法实时更新呢，你可以通过天气APP、搜索引擎或者当地的气象平台查询最新天气情况哦～ 希望你那里天气晴朗，心情也像天气一样明媚！😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 342, 'prompt_tokens': 94, 'total_tokens': 436, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 281, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'doubao-seed-1-6-flash-250828', 'system_fingerprint': None, 'id': '0217690786204068a58463ef17b425f3abf24ab17dedfd6325bb5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019be54d-7f1b-70b1-83c7-3e40ec97fc08-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 94, 'output_tokens': 342, 'total_tokens': 436, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 281}}


In [4]:
result.content

'你好呀！我叫豆包，是字节跳动研发的智能助手～  \n\n关于天气，我的知识没办法实时更新呢，你可以通过天气APP、搜索引擎或者当地的气象平台查询最新天气情况哦～ 希望你那里天气晴朗，心情也像天气一样明媚！😊'

In [6]:
from langchain.tools import tool

@tool("web_search", description="serch the query on the web")
def search_database(query: str, limit: int = 10) -> str:
    """search the database for the records matching the query

    Args:
        query: Serch term to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for the '{query}'"
print(search_database.description)

serch the query on the web


In [8]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for the weather query"""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )


weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast"""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 访问上下文

In [15]:
from langchain.tools import tool, ToolRuntime

@tool
def summarize_conversation(runtime: ToolRuntime) -> str:
    """summarize the conversation so far"""
    message = runtime.state("message")

    human_msgs = sum(1 for m in messages if m.__class__.__name__ == "HumanMessage")
    ai_msgs = sum(1 for m in messages if m.__class__.__name__ == "AIMessage")
    tool_msgs = sum(1 for m in messages if m.__class__.__name__ == "ToolMessage")

    return f"Conversation has {human_msgs} user messages, {ai_msgs} AI responses, and {tool_msgs} tool results"

# Access custom state fields
@tool
def get_user_preference(pref_name: str, runtime: ToolRuntime ) -> str:
    """get a user preference value"""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")



In [17]:
from langgraph.types import Command
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.tools import tool, ToolRuntime

# Update the conversation history by removing all messages
@tool
def clear_conversation() -> Command:
    """Clear the conversation history."""

    return Command(
        update = {
            "message": [RemoveMessage(id=REMOVE_ALL_MESSAGES)],
        }
    )

@tool
def update_user_name(
    new_name: str,
    runtime: ToolRuntime
) -> Command:
    """Update the user's name."""
    return Command(update={"user_name": new_name})

In [22]:
from dataclasses import dataclass
from langchain.agents import create_agent

USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"Account holder: {user['name']}\nType:{user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

model = llm
agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant."
)

result = agent.invoke(
    {"message": [{"role": "user", "content": "What is my current balance?"}]},
    context = UserContext(user_id="user456")
)

### 内存

In [40]:
from typing import Any
from langgraph.store.memory import InMemoryStore

# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

response = agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
